# UMI in MuJoCo - Phase 5: Train the diffusion policy on the simulated demos

Set **Runtime > Change runtime type > GPU** first. Run the cells in order.

**What this does:** trains UMI's own `DiffusionUnetTimmPolicy` (same code, same config file as the paper) on `sim_cup.zarr.zip` from Phase 4. Observations are the wrist-camera image and the gripper pose, the actions are 16 future relative gripper poses plus gripper width.

**What is changed from the paper's settings, and why**
- Image encoder: **ResNet-18** (ImageNet-pretrained) instead of CLIP ViT-B/16. The ViT-B has about 86 M parameters and was tested on a 24 GB RTX 3090; ResNet-18 has 11 M and fits a 16 GB T4 comfortably. Presets for others are in step 5.
- Batch size 64 and dataloader workers set to what Colab's CPUs can serve (Colab usually gives only 2 CPU cores, and every sample is JPEG-XL decoded).
- `task.obs_down_sample_steps=1`: the paper's GoPro data is 60 Hz and uses every 3rd frame; our demos are 10 Hz, so every frame is used.
- Weights & Biases is switched off; progress is read from the log file instead.
- Checkpoints go to Google Drive and **training resumes automatically** if Colab disconnects: just re-run steps 1-8.
- Two tiny patches to the repo's trainer (step 6): epoch counting on resume, and writing action-error numbers to the log. Both are checked by the notebook.

**Nothing here has been run on a real GPU by me.** Step 7 (a 2-epoch smoke test, takes a few minutes) exists to catch problems cheaply before you spend hours on the real run.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Paths and Google Drive

In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/umi_mujoco'
REPO = '/content/universal_manipulation_interface'
ENV  = '/content/umi_train_env'
PY   = f'{ENV}/bin/python'
DATA_DRIVE = f'{ROOT}/data/sim_cup.zarr.zip'
INFO_DRIVE = f'{ROOT}/data/sim_cup.info.json'
DATA = '/content/sim_cup.zarr.zip'          # local copy (reading a zip from Drive while training is slow)

for sub in ['checkpoints', 'logs']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)
assert os.path.exists(DATA_DRIVE), f'{DATA_DRIVE} not found - finish Phase 4 first'

os.environ.update(ROOT=ROOT, REPO=REPO, ENV=ENV, PY=PY, WANDB_MODE='disabled', PYTHONUNBUFFERED='1')
print('dataset on Drive: %.2f GB' % (os.path.getsize(DATA_DRIVE) / 1e9))
import shutil
tot, used, free = shutil.disk_usage('/content/drive')
print('Drive free space: %.1f GB (each checkpoint is roughly 0.5-1 GB; about 3 are kept)' % (free / 1e9))

## 3. Python 3.9 training environment (about 5 minutes, repeated in every new session)
Same pins as Phase 1 (with `av 12.3.0`), without `mujoco` and the system rendering libraries, which training does not need.

In [ ]:
%%bash
set -e
if [ ! -d "$REPO/.git" ]; then
  git clone https://github.com/real-stanford/universal_manipulation_interface.git "$REPO"
fi
cd "$REPO" && git log -1 --format='repo at %h (%ad)'

In [ ]:
%%writefile /content/umi_train_requirements.txt
torch==2.1.0
torchvision==0.16.0
numpy==1.24.4
scipy==1.11.4
numba==0.57.1
pandas==2.1.4
opencv-python-headless==4.7.0.72
matplotlib==3.7.5
av==12.3.0
pyyaml==6.0.1
tqdm==4.65.0
zarr==2.16.1
numcodecs==0.11.0
hydra-core==1.2.0
wandb==0.15.8
dill==0.3.7
einops==0.6.1
diffusers==0.18.2
huggingface_hub==0.19.4
scikit-video==1.1.11
lmdb
timm==0.9.7
scikit-fda==0.9
cython==0.29.37
accelerate==0.24.1
boto3
aiohttp==3.9.5
aiofiles==23.2.1
threadpoolctl==3.2.0
imagecodecs==2023.9.18
h5py
psutil
termcolor
tensorboardX
imageio
imageio-ffmpeg
filelock

In [ ]:
%%bash
set -eo pipefail
pip -q install uv
if [ -x "$PY" ]; then
  echo "environment already exists at $ENV (re-using)"
else
  uv python install 3.9
  uv venv "$ENV" --python 3.9 --seed
fi
uv pip install --python "$PY" -r /content/umi_train_requirements.txt 2>&1 | tail -15
uv pip install --python "$PY" --no-deps robomimic==0.2.0 2>&1 | tail -3
echo "install finished"

## 4. Verify the environment: GPU, libraries, and the training modules import

In [ ]:
%%bash
cd "$REPO"
export PYTHONPATH="$REPO"
"$PY" - <<'EOF'
import sys, torch, numpy, zarr, imagecodecs, diffusers, timm
print('python', sys.version.split()[0], '| torch', torch.__version__, '| gpu available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0), '| %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print('numpy', numpy.__version__, '| zarr', zarr.__version__, '| diffusers', diffusers.__version__, '| timm', timm.__version__)
print('jpegxl available:', imagecodecs.JPEGXL.available)
import diffusion_policy.workspace.train_diffusion_unet_image_workspace
import diffusion_policy.dataset.umi_dataset
import diffusion_policy.policy.diffusion_unet_timm_policy
print('training modules import OK')
EOF

Expected: `gpu available True`, `jpegxl available: True`, `training modules import OK`. If anything else appears, paste the output to me before going on.

## 5. Copy the dataset to local disk and settings

In [ ]:
import shutil, json, os
if not os.path.exists(DATA) or os.path.getsize(DATA) != os.path.getsize(DATA_DRIVE):
    print('copying dataset to local disk ...')
    shutil.copy(DATA_DRIVE, DATA)
print('local dataset: %.2f GB' % (os.path.getsize(DATA) / 1e9))

info = {}
if os.path.exists(INFO_DRIVE):
    info = json.load(open(INFO_DRIVE))
    print('episodes:', info.get('n_episodes', '?'), '| frames:', info.get('n_frames', '?'), '| record_dt:', info.get('record_dt', '?'))
dataset_overrides = info.get('training_overrides') or ['task.obs_down_sample_steps=1', 'task.dataset_frequeny=0']
print('dataset overrides:', dataset_overrides)

### Settings
Pick one **PRESET**. `resnet18` is the safe choice for a free T4. Lower `BATCH` (32, 16) if step 7 or 8 reports *CUDA out of memory*.

| preset | image encoder | notes |
|---|---|---|
| `resnet18` | ResNet-18, pretrained | default; fastest |
| `resnet34` | ResNet-34, pretrained | a bit slower, a bit more capacity |
| `vit_tiny` | ViT-Tiny, pretrained | transformer encoder, uses the CLS token |
| `clip_vit_b16` | CLIP ViT-B/16 (the paper's setting) | needs about 24 GB of GPU memory; will probably run out of memory on a T4 |

`EPOCHS`: one epoch is one pass over all training windows. The paper trains 120 epochs on far more data. 40 is a reasonable first run; I could not measure the speed here, so look at `it/s` on the progress bar in step 8 and compute: epoch time = steps per epoch / (it/s).

In [ ]:
PRESET = 'resnet18'
EPOCHS = 40
BATCH  = None          # None = use the preset's default
SEED   = 42

PRESETS = {
    'resnet18':     dict(model='resnet18.a1_in1k', batch=64),
    'resnet34':     dict(model='resnet34.a1_in1k', batch=32),
    'vit_tiny':     dict(model='vit_tiny_patch16_224', batch=64),
    'clip_vit_b16': dict(model='vit_base_patch16_clip_224.openai', batch=16),
}
p = PRESETS[PRESET]
BATCH = BATCH or p['batch']
NUM_WORKERS = max(1, min(4, os.cpu_count() or 2))
RUN_NAME = PRESET
RUN_DIR  = f'{ROOT}/checkpoints/{RUN_NAME}'

COMMON = [
    f'task.dataset_path={DATA}',
    *dataset_overrides,
    f'policy.obs_encoder.model_name={p["model"]}',
    'policy.obs_encoder.pretrained=True',
    f'dataloader.batch_size={BATCH}', f'val_dataloader.batch_size={BATCH}',
    f'dataloader.num_workers={NUM_WORKERS}', f'val_dataloader.num_workers={NUM_WORKERS}',
    f'training.seed={SEED}',
    'logging.mode=disabled',
    'training.tqdm_interval_sec=30',
]
TRAIN_ARGS = COMMON + [
    f'training.num_epochs={EPOCHS}',
    'training.resume=True',
    'training.checkpoint_every=4', 'training.sample_every=2', 'training.rollout_every=1000',
    'checkpoint.topk.k=2',
    f'hydra.run.dir={RUN_DIR}', f'hydra.sweep.dir={RUN_DIR}',
]
print('run folder (on Drive):', RUN_DIR)
print('workers:', NUM_WORKERS, '| batch:', BATCH, '| encoder:', p['model'], '| epochs:', EPOCHS)

## 6. Patch two lines of the repo's trainer
1. **Resume.** The trainer always loops `range(num_epochs)`, so after a resume it would train another full `num_epochs`. It becomes `range(self.epoch, num_epochs)`: identical on a fresh run, and a resumed run only does the remaining epochs.
2. **Log the action-error numbers.** The trainer computes validation action errors as tensors, which its JSON logger silently drops. They are converted to plain floats so the plot in step 9 can show them.

The cell asserts that each line is found exactly once, so it fails loudly if the repo changed.

In [ ]:
p = f'{REPO}/diffusion_policy/workspace/train_diffusion_unet_image_workspace.py'
s = open(p).read()

old1 = 'for local_epoch_idx in range(cfg.training.num_epochs):'
new1 = 'for local_epoch_idx in range(self.epoch, cfg.training.num_epochs):'
old2 = '''                accelerator.log(step_log, step=self.global_step)
                json_logger.log(step_log)
                self.global_step += 1
                self.epoch += 1'''
new2 = '''                step_log = {k: (v.item() if hasattr(v, 'item') else v) for k, v in step_log.items()}
                accelerator.log(step_log, step=self.global_step)
                json_logger.log(step_log)
                self.global_step += 1
                self.epoch += 1'''

if new1 in s:
    print('patch 1 already applied')
else:
    assert s.count(old1) == 1, 'patch 1: line not found exactly once'
    s = s.replace(old1, new1)
    print('patch 1 applied')
if "step_log = {k: (v.item()" in s:
    print('patch 2 already applied')
else:
    assert s.count(old2) == 1, 'patch 2: block not found exactly once'
    s = s.replace(old2, new2)
    print('patch 2 applied')
open(p, 'w').write(s)
compile(s, p, 'exec')
print('trainer file still compiles')

## 7. Smoke test (2 epochs x 3 steps, a few minutes)
Runs the real training script in the repo's own `debug` mode: it loads the dataset, builds the model, trains a few steps, samples actions from the policy, and writes checkpoints. Output goes to local disk, not Drive. The output appears when it finishes.

In [ ]:
import subprocess, shlex
SMOKE_DIR = '/content/smoke_run'
!rm -rf {SMOKE_DIR}
SMOKE_ARGS = COMMON + ['training.debug=True', 'training.resume=False', 'training.num_epochs=2',
                       f'hydra.run.dir={SMOKE_DIR}', f'hydra.sweep.dir={SMOKE_DIR}']
CMD = f'cd {REPO} && PYTHONPATH={REPO} WANDB_MODE=disabled {PY} train.py --config-name=train_diffusion_unet_timm_umi_workspace ' + ' '.join(SMOKE_ARGS)
!{CMD} 2>&1 | tee /content/smoke.log | tail -45

### 7b. Check the smoke test result

In [ ]:
import json, os, math
ok = True
log = open('/content/smoke.log', errors='replace').read()
for line in log.splitlines():
    if line.startswith('train dataset:') or line.startswith('val dataset:'):
        print(line)
if 'Traceback' in log:
    ok = False
    print('FAIL: the run raised an error - paste the last 40 lines above to me')
rows = []
lp = f'{SMOKE_DIR}/logs.json.txt'
if os.path.exists(lp):
    rows = [json.loads(l) for l in open(lp) if l.strip()]
losses = [r['train_loss'] for r in rows if 'train_loss' in r]
print('logged steps:', len(rows), '| losses:', ['%.3f' % x for x in losses])
if not losses or not all(math.isfinite(x) for x in losses):
    ok = False; print('FAIL: no finite training loss was logged')
ck = f'{SMOKE_DIR}/checkpoints/latest.ckpt'
if os.path.exists(ck):
    print('checkpoint written: %.0f MB' % (os.path.getsize(ck) / 1e6))
else:
    ok = False; print('FAIL: no checkpoint was written')
has_mse = any('val_action_mse_error_pos' in r for r in rows)
print('action-error rows in log:', has_mse)
print('\nSMOKE TEST:', 'PASS - go on to step 8' if ok else 'FAIL')

**Reading the numbers:** `train dataloader: N` is the number of steps per epoch. With the speed shown in step 8 (`it/s`), one epoch takes N / (it/s) seconds.

## 8. Train (resumes automatically)
Output (every 30 s) is also saved to a log file on Drive. Checkpoints are saved every 4 epochs into the run folder on Drive.

- **If the session disconnects or you stop the cell:** reconnect, re-run steps 2-6 with the same `PRESET` and `EPOCHS`, then run this cell again. It continues from the last checkpoint with the learning-rate schedule intact. The EMA weights restart averaging after a resume, which only costs a few hundred steps.
- **CUDA out of memory:** set a smaller `BATCH` in the settings cell and run this cell again (the run folder is named after the preset only, so it continues from the last checkpoint; changing the batch size is fine, changing the preset is not).
- Colab free sessions last at most about 12 hours, and may end sooner if the tab is closed.

In [ ]:
os.makedirs(RUN_DIR, exist_ok=True)
CMD = (f'cd {REPO} && PYTHONPATH={REPO} WANDB_MODE=disabled PYTHONUNBUFFERED=1 {PY} train.py '
       '--config-name=train_diffusion_unet_timm_umi_workspace ' + ' '.join(TRAIN_ARGS))
print('resuming' if os.path.exists(f'{RUN_DIR}/checkpoints/latest.ckpt') else 'starting fresh', '->', RUN_DIR)
!{CMD} 2>&1 | tee -a {ROOT}/logs/train_{RUN_NAME}.log

## 9. Training curves
Can be run at any time from a second tab or after training. Training loss is the noise-prediction error of the diffusion model; it drops fast, then flattens. The right plot shows the error of the policy's actual sampled actions, which is the more meaningful number.

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt

log_path = f'{RUN_DIR}/logs.json.txt'
if not os.path.exists(log_path):
    raise SystemExit('no log yet at ' + log_path)
rows = []
for line in open(log_path):
    line = line.strip()
    if not line:
        continue
    try:
        rows.append(json.loads(line))
    except json.JSONDecodeError:
        pass            # half-written last line
steps  = np.array([r['global_step'] for r in rows if 'train_loss' in r])
losses = np.array([r['train_loss']  for r in rows if 'train_loss' in r])
print('log lines:', len(rows), '| last epoch:', max(r.get('epoch', 0) for r in rows),
      '| last loss: %.4f' % losses[-1])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(steps, losses, alpha=0.25, label='per step')
k = min(100, max(1, len(losses) // 5))
if len(losses) > k:
    ax[0].plot(steps[k - 1:], np.convolve(losses, np.ones(k) / k, mode='valid'), label=f'moving avg ({k})')
ax[0].set_yscale('log'); ax[0].set_xlabel('step'); ax[0].set_title('training loss (noise prediction)'); ax[0].legend()

val = [(r['epoch'], r) for r in rows if 'val_action_mse_error_pos' in r]
if val:
    ep = [e for e, _ in val]
    ax[1].plot(ep, [r['val_action_mse_error_pos'] for _, r in val], 'o-', label='val position')
    ax[1].plot(ep, [r['train_action_mse_error_pos'] for _, r in val], 's--', label='train position')
    ax[1].set_xlabel('epoch'); ax[1].set_yscale('log'); ax[1].set_title('action error on a batch (lower = better)'); ax[1].legend()
else:
    ax[1].text(0.5, 0.5, 'no action-error rows yet', ha='center'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 10. Offline check on held-out demos
Loads the newest checkpoint, predicts action chunks for demos the model never saw (5 % of the data is held out), and compares them with what the scripted expert actually did.

How to read it: *MSE/var* is the error relative to always guessing the average action; it should be well below 1. The position errors tell you how far off the predicted path is (a cup is 5 cm wide, so an error of a few centimetres in the near future is the threshold of success or failure). This is **not** a success rate. That comes from running the policy in MuJoCo in Phase 6.

In [ ]:
%%writefile /content/offline_check.py
"""Offline check of a trained UMI policy: predict action chunks for held-out demos and compare with the real ones.

    python offline_check.py --ckpt latest.ckpt --dataset sim_cup.zarr.zip
Run inside the Python 3.9 training env with PYTHONPATH set to the UMI repo.
"""
import argparse, os, sys
import numpy as np
import torch, dill, hydra
from omegaconf import OmegaConf
from torch.utils.data import DataLoader

OmegaConf.register_new_resolver("eval", eval, replace=True)
from diffusion_policy.common.pytorch_util import dict_apply

ap = argparse.ArgumentParser()
ap.add_argument("--ckpt", required=True)
ap.add_argument("--dataset", required=True)
ap.add_argument("--batches", type=int, default=8)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--png", default="/content/offline_check.png")
a = ap.parse_args()

torch.manual_seed(0)
np.random.seed(0)
payload = torch.load(open(a.ckpt, "rb"), pickle_module=dill, map_location="cpu")
cfg = payload["cfg"]
# the checkpoint stores the resolved config, so the dataset path has to be replaced in both places
cfg.task.dataset_path = a.dataset
cfg.task.dataset.dataset_path = a.dataset

cls = hydra.utils.get_class(cfg._target_)
ws = cls(cfg, output_dir="/content/_offline_tmp")
ws.load_payload(payload, exclude_keys=["optimizer"])
policy = ws.ema_model if cfg.training.use_ema else ws.model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy.to(device).eval()
print("checkpoint epoch:", getattr(ws, "epoch", "?"), "| global_step:", getattr(ws, "global_step", "?"),
      "| policy:", "EMA" if cfg.training.use_ema else "raw")

dataset = hydra.utils.instantiate(cfg.task.dataset)
val = dataset.get_validation_dataset()
print("validation samples:", len(val))
dl = DataLoader(val, batch_size=a.bs, shuffle=True, num_workers=2)

preds, gts = [], []
with torch.no_grad():
    for i, batch in enumerate(dl):
        if i >= a.batches:
            break
        batch = dict_apply(batch, lambda x: x.to(device))
        pred = policy.predict_action(batch["obs"], None)["action_pred"]
        preds.append(pred.float().cpu().numpy())
        gts.append(batch["action"].float().cpu().numpy())
P = np.concatenate(preds)
G = np.concatenate(gts)
print("predicted action chunks:", P.shape, "(samples, horizon, 10 = pos3 + rot6d + width1)")

parts = {"pos (m)": slice(0, 3), "rot6d": slice(3, 9), "width (m)": slice(9, 10)}
print("\n%-10s %12s %16s %10s" % ("part", "MSE", "var of GT", "MSE/var"))
for name, sl in parts.items():
    mse = float(((P[..., sl] - G[..., sl]) ** 2).mean())
    var = float(G[..., sl].reshape(-1, G[..., sl].shape[-1]).var(axis=0).mean())
    print("%-10s %12.3e %16.3e %10.3f" % (name, mse, var, mse / max(var, 1e-12)))
print("(MSE/var is the error relative to always guessing the average action. Well below 1 = the policy learned something.)")

err = np.linalg.norm(P[..., :3] - G[..., :3], axis=-1)  # (N, T) metres
print("\nposition error of the predicted chunk (cm), by step into the future:")
for t in (0, 3, 7, 11, 15):
    if t < err.shape[1]:
        print("   step %2d: mean %.2f cm | 90th percentile %.2f cm" % (t, 100 * err[:, t].mean(), 100 * np.percentile(err[:, t], 90)))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
n = min(3, len(P))
fig, axes = plt.subplots(2, n, figsize=(4.2 * n, 6), squeeze=False)
for i in range(n):
    for d, c in zip(range(3), ("tab:red", "tab:green", "tab:blue")):
        axes[0, i].plot(G[i, :, d], "-", color=c, label="true " + "xyz"[d])
        axes[0, i].plot(P[i, :, d], "--", color=c, label="pred " + "xyz"[d])
    axes[0, i].set_title("held-out sample %d: position (relative to now)" % i, fontsize=9)
    axes[0, i].set_xlabel("step into the future")
    axes[1, i].plot(G[i, :, 9], "k-", label="true")
    axes[1, i].plot(P[i, :, 9], "k--", label="pred")
    axes[1, i].set_title("gripper width", fontsize=9)
    axes[1, i].set_xlabel("step into the future")
axes[0, 0].legend(fontsize=7, ncol=2)
axes[1, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(a.png, dpi=110)
print("\nsaved", a.png)

In [ ]:
CKPT = f'{RUN_DIR}/checkpoints/latest.ckpt'
assert os.path.exists(CKPT), 'no checkpoint yet - train for at least 4 epochs first'
!cd {REPO} && PYTHONPATH={REPO} WANDB_MODE=disabled {PY} /content/offline_check.py --ckpt {CKPT} --dataset {DATA}
from IPython.display import Image, display
display(Image('/content/offline_check.png'))

## Phase 5 checklist

- [ ] Step 4: GPU visible, `training modules import OK`
- [ ] Step 7b: `SMOKE TEST: PASS`
- [ ] Step 8: epochs complete, loss decreasing, checkpoints in `umi_mujoco/checkpoints/<run>/checkpoints/`
- [ ] Step 10: MSE/var well below 1 for position and width, predicted curves follow the true ones

**Send me:** the step 7b output, the step 8 `it/s` and the `train dataloader:` line, the final loss, and the step 10 table. If the position MSE/var is not clearly below 1, the fix is more epochs or more demos, not code.

Next: Phase 6 - the MuJoCo evaluation loop: the trained policy controls the simulated Panda, using the same relative-pose action conversion as `eval_real.py`.